# 03 · Dropout Prediction — Fabric Data Science

**Sprint 2, Task 2.2**

Trains a GradientBoostingClassifier to predict learner dropout risk.
All tracking via MLflow (auto-integrated with Fabric Data Science).

**Prerequisites:** Run notebook `01_xapi_ingestion.ipynb` first so that
`silver_xapi_events` exists with at least a few thousand rows.

**Output:** Model registered as `dropout_predictor` in the Fabric model registry.

## 0 · Setup

In [ ]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print(f"MLflow version: {mlflow.__version__}")
print("Ready.")

## 1 · Build feature matrix from silver_xapi_events

In [ ]:
df_raw = spark.sql("""
    SELECT
        learner_email,
        course_id,
        COUNT(*)                                                          AS total_events,
        COUNT(DISTINCT module_id)                                         AS modules_touched,
        ROUND(AVG(CASE WHEN verb = 'scored' THEN score ELSE NULL END), 4) AS avg_score,
        COUNT(DISTINCT CAST(ts AS DATE))                                  AS active_days,
        DATEDIFF(day, MIN(ts), MAX(ts))                                   AS span_days,
        MAX(CASE WHEN verb = 'completed' AND course_completed = true
                 THEN 1 ELSE 0 END)                                       AS completed
    FROM silver_xapi_events
    GROUP BY learner_email, course_id
    HAVING COUNT(*) >= 5
""").toPandas()

print(f"Total learner-course pairs: {len(df_raw):,}")
print(f"Completion rate: {df_raw['completed'].mean():.1%}")
df_raw.head()

In [ ]:
# ── Feature engineering ────────────────────────────────────────────────────────
df = df_raw.copy()

# Fill missing avg_score (learners who never triggered a scored event)
df['avg_score'] = df['avg_score'].fillna(df['avg_score'].median())

# Derived features
df['events_per_day'] = df['total_events'] / df['active_days'].clip(lower=1)
df['completion_velocity'] = df['modules_touched'] / df['span_days'].clip(lower=1)
df['score_x_events'] = df['avg_score'] * df['total_events']

FEATURE_COLS = [
    'total_events',
    'modules_touched',
    'avg_score',
    'active_days',
    'span_days',
    'events_per_day',
    'completion_velocity',
    'score_x_events',
]
LABEL_COL = 'completed'

X = df[FEATURE_COLS]
y = df[LABEL_COL]

print(f"Features: {FEATURE_COLS}")
print(f"Label distribution:\n{y.value_counts()}")
X.describe()

## 2 · Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Train: {len(X_train):,} rows  |  Test: {len(X_test):,} rows")
print(f"Train completion rate: {y_train.mean():.1%}")
print(f"Test  completion rate: {y_test.mean():.1%}")

## 3 · Train with MLflow tracking

In [ ]:
EXPERIMENT_NAME = "dropout_prediction"
mlflow.set_experiment(EXPERIMENT_NAME)

# Hyperparameters
GBM_PARAMS = dict(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    min_samples_leaf=20,
    random_state=42,
)

with mlflow.start_run(run_name="gbm_v1") as run:
    RUN_ID = run.info.run_id
    print(f"MLflow run: {RUN_ID}")

    # Build pipeline: no scaling needed for GBM, but keeps the pattern extensible
    model = GradientBoostingClassifier(**GBM_PARAMS)
    model.fit(X_train, y_train)

    # ── Predictions ────────────────────────────────────────────────────────
    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    # ── Metrics ────────────────────────────────────────────────────────────
    acc       = accuracy_score(y_test, y_pred)
    prec      = precision_score(y_test, y_pred, zero_division=0)
    rec       = recall_score(y_test, y_pred, zero_division=0)
    f1        = f1_score(y_test, y_pred, zero_division=0)
    roc_auc   = roc_auc_score(y_test, y_pred_proba)

    # Cross-validation
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
    cv_mean   = cv_scores.mean()
    cv_std    = cv_scores.std()

    # ── Log to MLflow ──────────────────────────────────────────────────────
    mlflow.log_params(GBM_PARAMS)
    mlflow.log_params({"features": str(FEATURE_COLS), "n_features": len(FEATURE_COLS)})

    mlflow.log_metrics({
        "accuracy":    acc,
        "precision":   prec,
        "recall":      rec,
        "f1":          f1,
        "roc_auc":     roc_auc,
        "cv_roc_auc_mean": cv_mean,
        "cv_roc_auc_std":  cv_std,
        "train_rows":  len(X_train),
        "test_rows":   len(X_test),
    })

    # Feature importances
    importances = dict(zip(FEATURE_COLS, model.feature_importances_))
    mlflow.log_metrics({f"fi_{k}": v for k, v in importances.items()})

    # Log the model
    mlflow.sklearn.log_model(
        model,
        artifact_path="dropout_model",
        input_example=X_test.iloc[:3],
    )

    print(f"\n{'='*50}")
    print(f"  Accuracy:   {acc:.3f}")
    print(f"  Precision:  {prec:.3f}")
    print(f"  Recall:     {rec:.3f}")
    print(f"  F1:         {f1:.3f}")
    print(f"  ROC-AUC:    {roc_auc:.3f}")
    print(f"  CV ROC-AUC: {cv_mean:.3f} +/- {cv_std:.3f}")
    print(f"{'='*50}")

print("Run complete.")

In [ ]:
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Dropout', 'Completed']))

print("\nConfusion Matrix (rows=actual, cols=predicted):")
print("                 Pred Dropout  Pred Completed")
cm = confusion_matrix(y_test, y_pred)
print(f"Actual Dropout       {cm[0][0]:>5}           {cm[0][1]:>5}")
print(f"Actual Completed     {cm[1][0]:>5}           {cm[1][1]:>5}")

In [ ]:
# Feature importance chart
fi_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(fi_df['feature'], fi_df['importance'], color='steelblue')
ax.set_xlabel('Importance')
ax.set_title('Feature Importances — Dropout Predictor')
plt.tight_layout()

# Save and log to MLflow
fig.savefig('/tmp/feature_importance.png', dpi=120)
with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_artifact('/tmp/feature_importance.png')

display(fig)
plt.close()

## 4 · Register model in Fabric model registry

In [ ]:
MODEL_NAME = "dropout_predictor"

model_uri = f"runs:/{RUN_ID}/dropout_model"
registered = mlflow.register_model(model_uri, MODEL_NAME)

print(f"Registered model:  {registered.name}")
print(f"Version:           {registered.version}")
print(f"Status:            {registered.status}")

## 5 · Inference example

In [ ]:
# Load from registry and score a sample batch
loaded_model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/latest")

sample = X_test.iloc[:10].copy()
sample['dropout_risk'] = loaded_model.predict_proba(sample)[:, 0]   # prob of NOT completing
sample['prediction'] = loaded_model.predict(sample[FEATURE_COLS])
sample['actual'] = y_test.iloc[:10].values

display(sample[FEATURE_COLS + ['dropout_risk', 'prediction', 'actual']].round(3))

In [ ]:
# Score all learner-course pairs and write back as a gold table
all_features = df[FEATURE_COLS].fillna(0)
df['dropout_risk_score'] = loaded_model.predict_proba(all_features)[:, 0]

at_risk = df[df['dropout_risk_score'] > 0.65][[
    'learner_email', 'course_id', 'dropout_risk_score',
    'modules_touched', 'avg_score', 'days_since_active'
]].sort_values('dropout_risk_score', ascending=False) if 'days_since_active' in df.columns else \
df[df['dropout_risk_score'] > 0.65][[
    'learner_email', 'course_id', 'dropout_risk_score',
    'modules_touched', 'avg_score'
]].sort_values('dropout_risk_score', ascending=False)

print(f"High-risk learner-course pairs (risk > 65%): {len(at_risk):,}")
display(at_risk.head(20))

# Persist as a gold table for the dashboard
spark_df = spark.createDataFrame(df[['learner_email', 'course_id', 'dropout_risk_score'] + FEATURE_COLS])
(
    spark_df
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('gold_dropout_risk')
)
print("gold_dropout_risk table written.")